### RAG Pipeline- DATA Ingestion to vector DB Pipeline

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from pathlib import Path


d:\Desktop\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 1 PDF files to process

Processing: HISTORY_OF_INDIA_FROM_THE_EARLIEST_TIME_122_AD.pdf
  ✓ Loaded 308 pages

Total documents loaded: 308


In [3]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs


In [4]:
chunks=split_documents(all_pdf_documents)
chunks

Split 308 documents into 1014 chunks

Example chunk:
Content: B.A., First Year
History, Paper - I
HISTORY OF INDIA FROM
EARLIEST TIME TO
1200 A.D.
e/;çns'k Hkkst ¼eqä½ fo'ofo|ky; & Hkksiky
MADHYA PRADESH BHOJ (OPEN) UNIVERSITY – BHOPAL...
Metadata: {'producer': 'PyPDF', 'creator': 'Nitro Pro 10', 'creationdate': '', 'moddate': '2022-03-09T15:14:57+05:30', 'source': '..\\data\\pdf\\HISTORY_OF_INDIA_FROM_THE_EARLIEST_TIME_122_AD.pdf', 'total_pages': 308, 'page': 0, 'page_label': '1', 'source_file': 'HISTORY_OF_INDIA_FROM_THE_EARLIEST_TIME_122_AD.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'PyPDF', 'creator': 'Nitro Pro 10', 'creationdate': '', 'moddate': '2022-03-09T15:14:57+05:30', 'source': '..\\data\\pdf\\HISTORY_OF_INDIA_FROM_THE_EARLIEST_TIME_122_AD.pdf', 'total_pages': 308, 'page': 0, 'page_label': '1', 'source_file': 'HISTORY_OF_INDIA_FROM_THE_EARLIEST_TIME_122_AD.pdf', 'file_type': 'pdf'}, page_content="B.A., First Year\nHistory, Paper - I\nHISTORY OF INDIA FROM\nEARLIEST TIME TO\n1200 A.D.\ne/;çns'k Hkkst ¼eqä½ fo'ofo|ky; & Hkksiky\nMADHYA PRADESH BHOJ (OPEN) UNIVERSITY – BHOPAL"),
 Document(metadata={'producer': 'PyPDF', 'creator': 'Nitro Pro 10', 'creationdate': '', 'moddate': '2022-03-09T15:14:57+05:30', 'source': '..\\data\\pdf\\HISTORY_OF_INDIA_FROM_THE_EARLIEST_TIME_122_AD.pdf', 'total_pages': 308, 'page': 1, 'page_label': '2', 'source_file': 'HISTORY_OF_INDIA_FROM_THE_EARLIEST_TIME_122_AD.pdf', 'file_type': 'pdf'}, page_content='Reviewer Committee\n1. Dr. Amita Singh\nProfessor,\nGovt. MLB College, Bhopal (MP).\n3. Dr. Mam

Embedding and vectorStoreDB

In [5]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [6]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str="all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name=model_name
        self.model=None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
    

## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager
   

Loading embedding model: all-MiniLM-L6-v2
Model loaded successfully. Embedding dimension: 384


vector store DB

In [7]:
from typing import List,Any

In [8]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 14831


In [9]:
chunks

[Document(metadata={'producer': 'PyPDF', 'creator': 'Nitro Pro 10', 'creationdate': '', 'moddate': '2022-03-09T15:14:57+05:30', 'source': '..\\data\\pdf\\HISTORY_OF_INDIA_FROM_THE_EARLIEST_TIME_122_AD.pdf', 'total_pages': 308, 'page': 0, 'page_label': '1', 'source_file': 'HISTORY_OF_INDIA_FROM_THE_EARLIEST_TIME_122_AD.pdf', 'file_type': 'pdf'}, page_content="B.A., First Year\nHistory, Paper - I\nHISTORY OF INDIA FROM\nEARLIEST TIME TO\n1200 A.D.\ne/;çns'k Hkkst ¼eqä½ fo'ofo|ky; & Hkksiky\nMADHYA PRADESH BHOJ (OPEN) UNIVERSITY – BHOPAL"),
 Document(metadata={'producer': 'PyPDF', 'creator': 'Nitro Pro 10', 'creationdate': '', 'moddate': '2022-03-09T15:14:57+05:30', 'source': '..\\data\\pdf\\HISTORY_OF_INDIA_FROM_THE_EARLIEST_TIME_122_AD.pdf', 'total_pages': 308, 'page': 1, 'page_label': '2', 'source_file': 'HISTORY_OF_INDIA_FROM_THE_EARLIEST_TIME_122_AD.pdf', 'file_type': 'pdf'}, page_content='Reviewer Committee\n1. Dr. Amita Singh\nProfessor,\nGovt. MLB College, Bhopal (MP).\n3. Dr. Mam

In [10]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the embeddings
embddings=embedding_manager.generate_embeddings(texts)

## store in the vector database
vectorstore.add_documents(chunks,embddings)

Generating embeddings for 1014 texts...


Batches: 100%|██████████| 32/32 [00:22<00:00,  1.42it/s]


Generated embeddings with shape: (1014, 384)
Adding 1014 documents to vector store...
Successfully added 1014 documents to vector store
Total documents in collection: 15845


Retriever Pipeline From VectorStore

In [11]:
from typing import List, Dict, Any

class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store, embedding_manager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        print(f"Retrieving documents for query: '{query}'")
        print(f"top K: {top_k}, Score threshold: {score_threshold}")

        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(
                        zip(ids, documents, metadatas, distances)):

                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })

                print(f"Retrieved {len(retrieved_docs)} documents")
            else:
                print("No documents found")

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []
rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [12]:
rag_retriever

In [13]:
rag_retriever.retrieve("explain the concept of machine learning", top_k=3, score_threshold=0.5)

Retrieving documents for query: 'explain the concept of machine learning'
top K: 3, Score threshold: 0.5
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 25.64it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents


[]

RAG Pipeline- VectorDB To LLM Output Generation

In [14]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from dotenv import load_dotenv
import os

load_dotenv()

llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Meta-Llama-3-8B-Instruct",
    task="conversational",
    # token=os.getenv("HUGGINGFACEHUB_API_TOKEN")
)

model = ChatHuggingFace(llm=llm)

def rag_simple(query, retriever, model, top_k=3):
    # retrieve context
    results = retriever.retrieve(query, top_k=top_k)
    context = "\n\n".join(doc['content'] for doc in results) if results else ""

    if not context:
        return "No relevant context found to answer the question."

    prompt = f"""Use the following context to answer the question concisely.

Context:
{context}

Question: {query}

Answer:
"""

    response = model.invoke(prompt)
    return response.content




# chat_history=[]
# while True:
#     user_input = input("You :- ")
#     chat_history.append(user_input)
#     if user_input == "exit":
#         break
#     result = model.invoke(chat_history)
#     chat_history.append(result)
#     print("AI :- ", result.content if hasattr(result, "content") else result)




In [15]:
answer=rag_simple("explain indian history?",rag_retriever,model)
print(answer)

Retrieving documents for query: 'explain indian history?'
top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 195.15it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents


Indian History can be broadly classified into several themes, including:

1. **Indus Valley Civilization**: Harappan and Mohenjo-Daro culture (3300-1300 BCE)
2. **Vedic Period**: Emergence of Hinduism and Vedic culture (1500-500 BCE)
3. **Mauryan and Gupta Empires**: Rise of imperial India (321 BCE-550 CE)
4. **Medieval Period**: Muslim invasions, Delhi Sultanate, and Mughal Empire (1206-1857 CE)
5. **Colonial Era**: British East India Company, British Raj, and Indian National Movement (1757-1947 CE)
6. **Modern India**: Post-independence, partition, and contemporary India (1947 CE-present)
7. **Regional and Social Variations**: Regional histories, social reforms, and cultural developments.

These themes provide a general overview of the complex and diverse history of India.
